In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

PROJECT_ROOT = Path('../../')
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
BASELINE_TAG = 'BASELINE_20250815T000000Z'

BASINS = ['calpella', 'guerneville', 'hopland', 'warm_springs']
RESOLUTIONS = ['daily', 'hourly', 'mts_daily', 'mts_hourly']
MODELS = ['HMS', 'LSTM', 'PILSTM']

In [ ]:
def get_metrics_path(basin, res):
    base = OUTPUTS_DIR / basin / res / BASELINE_TAG / 'metrics' / 'test'
    if res in ('daily', 'hourly'):
        return base / f'{basin}_{res}_test_metrics.csv'
    freq = '1D' if res == 'mts_daily' else '1H'
    return base / f'{basin}_mts_test_metrics_{freq}.csv'

rows = []
for basin in BASINS:
    for res in RESOLUTIONS:
        path = get_metrics_path(basin, res)
        if not path.exists():
            print(f"MISSING: {path}")
            continue
        df = pd.read_csv(path, index_col=0)
        for metric in df.index:
            if metric == 'MSE':  # redundant with RMSE
                continue
            for model in MODELS:
                rows.append({'basin': basin, 'resolution': res, 'model': model,
                             'metric': metric, 'value': pd.to_numeric(df.loc[metric, model], errors='coerce')})

metrics_df = pd.DataFrame(rows)
print(f"Loaded {len(metrics_df)} entries | {metrics_df.basin.nunique()} basins | {metrics_df.resolution.nunique()} resolutions")
print(f"Metrics: {metrics_df.metric.unique().tolist()}")

In [ ]:
metrics_df

In [ ]:
def pivot_basin(basin):
    df = metrics_df[metrics_df['basin'] == basin].pivot(index='metric', columns=['resolution', 'model'], values='value').round(3)
    df.columns = [f'{res}_{mod}' for res, mod in df.columns]
    return df

In [ ]:
calpella_df = pivot_basin('calpella')
guerneville_df = pivot_basin('guerneville')
hopland_df = pivot_basin('hopland')
warm_springs_df = pivot_basin('warm_springs')

calpella_df

### Metrics Description Summary

#### Overall Performance

| Metric | Formula | Optimal | Interpretation |
|---|---|---|---|
| **NSE** | $1 - \frac{\sum(Q_{sim} - Q_{obs})^2}{\sum(Q_{obs} - \bar{Q}_{obs})^2}$ | 1 (higher = better) | How much better than predicting the mean. NSE=0 means no better than the mean; NSE<0 means worse. |
| **KGE** | $1 - \sqrt{s_r(r-1)^2 + s_\alpha(\alpha-1)^2 + s_\beta(\beta_{KGE}-1)^2}$ | 1 (higher = better) | Composite of correlation ($r$), variability ratio ($\alpha$), and bias ratio ($\beta$) with weights $s_r, s_\alpha, s_\beta$ (default 1,1,1). Addresses NSE's known shortcomings. |
| **Pearson-r** | $\frac{\sum(Q_{obs} - \bar{Q}_{obs})(Q_{sim} - \bar{Q}_{sim})}{\sigma_{obs} \cdot \sigma_{sim}}$ | 1 (higher = better) | Pure correlation — captures timing/shape. Ignores magnitude and bias (means are subtracted out). |
| **RMSE** | $\sqrt{\frac{1}{T}\sum(\hat{y}_t - y_t)^2}$ | 0 (lower = better) | Average error magnitude in flow units (cfs). Penalizes large errors heavily. |

#### Bias Metrics

| Metric | Formula | Optimal | Interpretation |
|---|---|---|---|
| **PBIAS** | $\frac{\sum(Q_{sim} - Q_{obs})}{\sum Q_{obs}} \times 100$ | 0 (closer to 0) | Percent volume bias. Positive = overprediction, negative = underprediction. |
| **Beta-NSE** | $\frac{\mu_{sim} - \mu_{obs}}{\sigma_{obs}}$ | 0 (closer to 0) | Normalized bias. Same concept as PBIAS but scaled by observed variability. |
| **Beta-KGE** | $\frac{\mu_{sim}}{\mu_{obs}}$ | 1 (closer to 1) | Mean ratio. >1 = overprediction, <1 = underprediction. |
| **Alpha-NSE** | $\frac{\sigma_{sim}}{\sigma_{obs}}$ | 1 (closer to 1) | Variability ratio. >1 = model too flashy, <1 = model too smooth/damped. |

#### Flow Duration Curve (FDC) Metrics

| Metric | What it Captures | Optimal | Interpretation |
|---|---|---|---|
| **FHV** | Bias in **top 2%** of flows | 0% (closer to 0) | Peak/flood volume error. Positive = overpredicting peaks, negative = underpredicting. |
| **FMS** | Slope error in **20th-70th percentile** | 0% (closer to 0) | Mid-range flow distribution error. |
| **FLV** | Bias in **bottom 30%** of flows (log-transformed) | 0% (closer to 0) | Low flow / baseflow error. Log transform so small values matter. |

#### Peak Metrics

| Metric | Formula | Optimal | Interpretation |
|---|---|---|---|
| **Peak-Timing** | Mean abs time difference between obs/sim peaks | 0 (lower = better) | How well peaks are aligned in time (days or hours). |
| **Peak-MAPE** | $\frac{1}{P}\sum\left\|\frac{Q_{s,p} - Q_{o,p}}{Q_{o,p}}\right\| \times 100$ | 0% (lower = better) | Mean absolute percentage error of peak magnitudes. |

#### Color Direction for Heatmaps

| Category | Metrics | Color Logic |
|---|---|---|
| Higher = better | NSE, KGE, Pearson-r | z-score as-is |
| Lower = better | RMSE, Peak-Timing, Peak-MAPE | negate z-score |
| Closer to 0 = better | PBIAS, Beta-NSE, FHV, FMS, FLV | z-score of $-|value|$ |
| Closer to 1 = better | Alpha-NSE, Beta-KGE | z-score of $-|value - 1|$ |

In [ ]:
HIGHER_IS_BETTER = ['NSE', 'KGE', 'Pearson-r']
LOWER_IS_BETTER = ['RMSE', 'Peak-Timing', 'Peak-MAPE']
CLOSER_TO_ZERO = ['PBIAS', 'Beta-NSE', 'FHV', 'FMS', 'FLV']
CLOSER_TO_ONE = ['Alpha-NSE', 'Beta-KGE']

METRIC_ORDER = HIGHER_IS_BETTER + CLOSER_TO_ONE + LOWER_IS_BETTER + CLOSER_TO_ZERO

In [ ]:
def transform_for_color(df):
    t = df.copy()
    for m in LOWER_IS_BETTER:
        if m in t.index: t.loc[m] = -t.loc[m]
    for m in CLOSER_TO_ZERO:
        if m in t.index: t.loc[m] = -t.loc[m].abs()
    for m in CLOSER_TO_ONE:
        if m in t.index: t.loc[m] = -(t.loc[m] - 1).abs()
    return t

In [ ]:
def ranking_summary(basin_df, basin, freq='daily'):
    if freq == 'daily':
        cols = {'HMS': 'daily_HMS', 'LSTM': 'daily_LSTM', 'PILSTM': 'daily_PILSTM', 'MTS_LSTM': 'mts_daily_LSTM', 'MTS_PILSTM': 'mts_daily_PILSTM'}
    else:
        cols = {'HMS': 'hourly_HMS', 'LSTM': 'hourly_LSTM', 'PILSTM': 'hourly_PILSTM', 'MTS_LSTM': 'mts_hourly_LSTM', 'MTS_PILSTM': 'mts_hourly_PILSTM'}
    subset = basin_df[list(cols.values())].copy()
    subset.columns = list(cols.keys())
    ordered = subset.reindex([m for m in METRIC_ORDER if m in subset.index])
    transformed = transform_for_color(ordered)
    winners = transformed.idxmax(axis=1)
    print(f"{basin.upper()} ({freq})")
    for model in cols.keys():
        won = winners[winners == model].index.tolist()
        if won:
            print(f"{model} - {len(won)} wins - {won}")

In [ ]:
FIGURES_DIR = Path('metrics_analysis/figures')
for sub in ['heatmaps', 'bar_charts', 'scatterplots', 'fdcs', 'hydrographs']:
    (FIGURES_DIR / sub).mkdir(parents=True, exist_ok=True)

def normalized_heatmap(basin_df, basin, freq='daily', figsize=(12, 8)):
    if freq == 'daily':
        cols = {'HMS': 'daily_HMS', 'LSTM': 'daily_LSTM', 'PILSTM': 'daily_PILSTM',
                'MTS_LSTM': 'mts_daily_LSTM', 'MTS_PILSTM': 'mts_daily_PILSTM'}
    else:
        cols = {'HMS': 'hourly_HMS', 'LSTM': 'hourly_LSTM', 'PILSTM': 'hourly_PILSTM',
                'MTS_LSTM': 'mts_hourly_LSTM', 'MTS_PILSTM': 'mts_hourly_PILSTM'}
    subset = basin_df[list(cols.values())].copy()
    subset.columns = list(cols.keys())
    ordered = subset.reindex([m for m in METRIC_ORDER if m in subset.index])
    transformed = transform_for_color(ordered)
    z = transformed.apply(lambda row: (row - row.mean()) / row.std(), axis=1)
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(z, annot=ordered.values, fmt='.2f', cmap='RdYlGn', center=0,
                ax=ax, cbar=False, annot_kws={'size': 10})
    ax.set_title(f"{basin.replace('_', ' ').title()} ({freq})")
    ax.set_ylabel('')
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'heatmaps' / f'{basin}_{freq}_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
normalized_heatmap(calpella_df, 'calpella', 'daily')

In [ ]:
ranking_summary(calpella_df, 'calpella', freq='daily')

### Calpella Daily Observations

#### MTS_PILSTM and PILSTM performs the best overall (6 and 5 wins) - physics is helping
##### - ...

In [ ]:
normalized_heatmap(calpella_df, 'calpella', 'hourly')

In [ ]:
ranking_summary(calpella_df, 'calpella', freq='hourly')

### Calpella Hourly Observations

##### Standalone PILSTM has the most "wins" - normal since it's fed pure hourly data? Performing well with the bias metrics - physics correcting model 

In [ ]:
normalized_heatmap(guerneville_df, 'guerneville', 'daily')

In [ ]:
ranking_summary(guerneville_df, 'guerneville', freq='daily')

### Guerneville Daily Observations

In [ ]:
normalized_heatmap(guerneville_df, 'guerneville', 'hourly')

In [ ]:
ranking_summary(guerneville_df, 'guerneville', freq='hourly')

### Guerneville Hourly Observations

In [ ]:
normalized_heatmap(hopland_df, 'hopland', 'daily')

In [ ]:
ranking_summary(hopland_df, 'hopland', freq='daily')

### Hopland Daily Observations

In [ ]:
normalized_heatmap(hopland_df, 'hopland', 'hourly')

In [ ]:
ranking_summary(hopland_df, 'hopland', freq='hourly')

### Hopland Hourly Observations

In [ ]:
normalized_heatmap(warm_springs_df, 'warm_springs', 'daily')

In [ ]:
ranking_summary(warm_springs_df, 'warm_springs', freq='daily')

### Warm Springs Daily Observations

In [ ]:
normalized_heatmap(warm_springs_df, 'warm_springs', 'hourly')

In [ ]:
ranking_summary(warm_springs_df, 'warm_springs', freq='hourly')

### Warm Springs Hourly Observations

### Some Overall Trends

In [ ]:
COLS_MAP = {
    'daily': {'HMS': 'daily_HMS', 'LSTM': 'daily_LSTM', 'PILSTM': 'daily_PILSTM', 'MTS_LSTM': 'mts_daily_LSTM', 'MTS_PILSTM': 'mts_daily_PILSTM'},
    'hourly': {'HMS': 'hourly_HMS', 'LSTM': 'hourly_LSTM', 'PILSTM': 'hourly_PILSTM', 'MTS_LSTM': 'mts_hourly_LSTM', 'MTS_PILSTM': 'mts_hourly_PILSTM'}}

def get_winners(basin_df, freq):
    cols = COLS_MAP[freq]
    subset = basin_df[list(cols.values())].copy()
    subset.columns = list(cols.keys())
    ordered = subset.reindex([m for m in METRIC_ORDER if m in subset.index])
    transformed = transform_for_color(ordered)
    return transformed.idxmax(axis=1)

In [ ]:
all_winners = []
basin_dfs = {'calpella': calpella_df, 'guerneville': guerneville_df,
             'hopland': hopland_df, 'warm_springs': warm_springs_df}
for basin, df in basin_dfs.items():
    for freq in ['daily', 'hourly']:
        winners = get_winners(df, freq)
        for metric, model in winners.items():
            all_winners.append({'basin': basin, 'freq': freq, 'metric': metric, 'winner': model})

winners_df = pd.DataFrame(all_winners)

In [ ]:
winners_df

In [ ]:
print("TOTAL WINS BY MODEL:")
print(winners_df['winner'].value_counts().to_string())

print("\nWINS BY MODEL AND FREQUENCY:")
print(winners_df.groupby(['freq', 'winner']).size().unstack(fill_value=0).to_string())

In [ ]:
print("MOST FREQUENT WINNER PER METRIC (across 8 basin-freq combos)")
for metric in METRIC_ORDER:
    sub = winners_df[winners_df['metric'] == metric]
    if len(sub) == 0:
        continue
    counts = sub['winner'].value_counts()
    print(f"  {metric}: {counts.to_dict()}")

In [ ]:
winners_df['has_physics'] = winners_df['winner'].isin(['PILSTM', 'MTS_PILSTM'])
physics_by_freq = winners_df.groupby(['freq', 'has_physics']).size().unstack(fill_value=0)
physics_by_freq.columns = ['No Physics', 'Physics']
print("PHYSICS vs NON-PHYSICS BY FREQUENCY")
print(physics_by_freq.to_string())
print(f"\nOverall: Physics {winners_df['has_physics'].sum()} / {len(winners_df)} ({winners_df['has_physics'].mean()*100:.0f}%)")

In [ ]:
winners_ml = winners_df[winners_df['winner'] != 'HMS'].copy()
winners_ml['is_mts'] = winners_ml['winner'].str.startswith('MTS')
mts_by_freq = winners_ml.groupby(['freq', 'is_mts']).size().unstack(fill_value=0)
mts_by_freq.columns = ['Standalone', 'MTS']
print("MTS vs STANDALONE (ML models only)")
print(mts_by_freq.to_string())

In [ ]:
print("WINS BY BASIN AND MODEL")
print(winners_df.groupby(['basin', 'winner']).size().unstack(fill_value=0).to_string())

In [ ]:
print("HMS WINS DETAIL")
hms_wins = winners_df[winners_df['winner'] == 'HMS']
if len(hms_wins) > 0:
    print(f"Total: {len(hms_wins)}")
    print(f"Metrics won: {hms_wins['metric'].value_counts().to_dict()}")
    print(f"By basin: {hms_wins['basin'].value_counts().to_dict()}")
    print(f"By freq: {hms_wins['freq'].value_counts().to_dict()}")

### Overall Trend Findings

### Margin Analysis

In [ ]:
def compute_margins(basin_df, freq):
    cols = COLS_MAP[freq]
    subset = basin_df[list(cols.values())].copy()
    subset.columns = list(cols.keys())
    ordered = subset.reindex([m for m in METRIC_ORDER if m in subset.index])
    transformed = transform_for_color(ordered)
    rows = []
    for metric in transformed.index:
        vals = transformed.loc[metric].sort_values(ascending=False)
        spread = vals.iloc[0] - vals.iloc[-1]
        pct = ((vals.iloc[0] - vals.iloc[1]) / spread * 100) if spread > 0 else 0
        rows.append({'metric': metric, 'winner': vals.index[0], 'runner_up': vals.index[1], 'pct_margin': round(pct, 1)})
    return pd.DataFrame(rows).set_index('metric')

for basin in BASINS:
    for freq in ['daily', 'hourly']:
        m = compute_margins(basin_dfs[basin], freq)
        close = m[m['pct_margin'] < 20]
        decisive = m[m['pct_margin'] > 50]
        print(f"{basin.upper()} ({freq})")
        print(m.to_string())
        if len(decisive) > 0:
            print(f"\nDecisive wins (>50%): {decisive.index.tolist()}")
        if len(close) > 0:
            print(f"Close calls (<20%): {close.index.tolist()}")

### Timeseries Analysis

In [ ]:
def get_ts_path(basin, res):
    base = OUTPUTS_DIR / basin / res / BASELINE_TAG / 'timeseries' / 'test'
    if res == 'daily': return base / f'{basin}_daily_test_combined_ts.csv'
    if res == 'hourly': return base / f'{basin}_hourly_test_combined_ts.csv'
    if res == 'mts_daily': return base / f'{basin}_mts_test_1D_combined_ts.csv'
    return base / f'{basin}_mts_test_1H_combined_ts.csv'

def load_ts(basin, res):
    df = pd.read_csv(get_ts_path(basin, res), parse_dates=['Date'], index_col='Date')
    df.columns = ['Observed', 'HMS', 'LSTM', 'PILSTM']
    return df

def load_all_models(basin, freq='daily'):
    stand_res, mts_res = ('daily', 'mts_daily') if freq == 'daily' else ('hourly', 'mts_hourly')
    stand, mts = load_ts(basin, stand_res), load_ts(basin, mts_res)
    result = stand[['Observed', 'HMS', 'LSTM', 'PILSTM']].copy()
    result['MTS_LSTM'] = mts['LSTM']
    result['MTS_PILSTM'] = mts['PILSTM']
    return result

MODEL_COLORS = {'HMS': '#1f77b4', 'LSTM': '#ff7f0e', 'PILSTM': '#2ca02c', 'MTS_LSTM': '#d62728', 'MTS_PILSTM': '#9467bd'}
ALL_MODELS = ['HMS', 'LSTM', 'PILSTM', 'MTS_LSTM', 'MTS_PILSTM']

#### Observed vs Predicted Scatterplots

In [ ]:
def scatter_obs_vs_pred(basin, freq='daily', figsize=(20, 4)):
    df = load_all_models(basin, freq)
    all_vals = pd.concat([df['Observed']] + [df[m] for m in ALL_MODELS]).dropna()
    lim = all_vals.quantile(0.995) * 1.05
    alpha = 0.3 if freq == 'daily' else 0.1
    s = 8 if freq == 'daily' else 2
    fig, axes = plt.subplots(1, 5, figsize=figsize)
    for ax, model in zip(axes, ALL_MODELS):
        valid = df[['Observed', model]].dropna()
        obs, pred = valid['Observed'], valid[model]
        ax.scatter(obs, pred, alpha=alpha, s=s, color=MODEL_COLORS[model])
        ax.plot([0, lim], [0, lim], 'k--', lw=0.8, alpha=0.5)
        ax.set_xlim(0, lim); ax.set_ylim(0, lim)
        ax.set_xlabel('Observed'); ax.set_ylabel('Predicted')
        r = obs.corr(pred)
        n_clipped = ((obs > lim) | (pred > lim)).sum()
        title = f'{model}  r={r:.3f}'
        if n_clipped > 0: title += f'  ({n_clipped} clipped)'
        ax.set_title(title, fontsize=10)
        ax.set_aspect('equal')
    fig.suptitle(f'{basin.replace("_", " ").title()} ({freq})', fontsize=14)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'scatterplots' / f'{basin}_{freq}_scatter.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
scatter_obs_vs_pred('calpella', 'daily')

In [ ]:
scatter_obs_vs_pred('calpella', 'hourly')

In [ ]:
scatter_obs_vs_pred('guerneville', 'daily')

In [ ]:
scatter_obs_vs_pred('guerneville', 'hourly')

In [ ]:
scatter_obs_vs_pred('hopland', 'daily')

In [ ]:
scatter_obs_vs_pred('hopland', 'hourly')

In [ ]:
scatter_obs_vs_pred('warm_springs', 'daily')

In [ ]:
scatter_obs_vs_pred('warm_springs', 'hourly')

#### Flow Duration Curves

In [ ]:
def flow_duration_curve(basin, freq='daily', figsize=(10, 6)):
    df = load_all_models(basin, freq)
    fig, ax = plt.subplots(figsize=figsize)
    for col in ['Observed'] + ALL_MODELS:
        vals = df[col].dropna().sort_values(ascending=False).values
        exceed = np.arange(1, len(vals) + 1) / len(vals) * 100
        kw = {'lw': 2.5, 'color': 'black', 'zorder': 5} if col == 'Observed' else {'lw': 1.2, 'color': MODEL_COLORS[col]}
        ax.plot(exceed, vals, label=col, **kw)
    ax.set_yscale('log')
    ax.set_xlabel('Exceedance Probability (%)')
    ax.set_ylabel('Flow (cfs)')
    ax.set_title(f'{basin.replace("_", " ").title()} FDC ({freq})')
    ax.legend()
    ylim = ax.get_ylim()
    for pct, lbl in [(2, 'FHV 2%'), (20, '20%'), (70, '70%')]:
        ax.axvline(x=pct, color='gray', ls=':', lw=0.6, alpha=0.4)
        ax.text(pct + 0.5, ylim[1] * 0.7, lbl, fontsize=7, color='gray')
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'fdcs' / f'{basin}_{freq}_fdc.png', dpi=300, bbox_inches='tight')
    plt.show()

for basin in BASINS:
    for freq in ['daily', 'hourly']:
        flow_duration_curve(basin, freq)

#### Hydrograph Timeseries — Top Peak Events

In [ ]:
def hydrograph(basin, freq='daily', top_n=3, figsize=(14, 12)):
    df = load_all_models(basin, freq)
    window = pd.Timedelta(days=30 if freq == 'daily' else 7)
    obs = df['Observed'].sort_values(ascending=False)
    peaks = []
    for date in obs.index:
        if not any(abs((date - p).total_seconds()) < window.total_seconds() for p in peaks):
            peaks.append(date)
            if len(peaks) >= top_n:
                break
    fig, axes = plt.subplots(top_n, 1, figsize=figsize, sharex=False)
    if top_n == 1: axes = [axes]
    half = window / 2
    for ax, peak in zip(axes, peaks):
        sub = df.loc[peak - half:peak + half]
        ax.plot(sub.index, sub['Observed'], 'k-', lw=2, label='Observed')
        for model in ALL_MODELS:
            ax.plot(sub.index, sub[model], lw=1, color=MODEL_COLORS[model], alpha=0.8, label=model)
        ax.set_ylabel('Flow (cfs)')
        ax.legend(fontsize=8, ncol=6, loc='upper right')
        ax.set_title(f'Peak: {peak.strftime("%Y-%m-%d")}')
    fig.suptitle(f'{basin.replace("_", " ").title()} Top {top_n} Events ({freq})', fontsize=14)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'hydrographs' / f'{basin}_{freq}_hydrograph.png', dpi=300, bbox_inches='tight')
    plt.show()

for basin in BASINS:
    for freq in ['daily', 'hourly']:
        hydrograph(basin, freq)